# Intro 03 — Probability: Axioms and Basic Rules

Practice notebook for the **"Probability: Axioms and Basic Rules"** section of the stats course PDF.

## What you will learn

This notebook recaps the theory from the PDF section, then **verifies it with Python**.
Each part ends with a **"Your turn"** prompt; the full **Problem Set** at the end has
**click-to-reveal solutions**.

## How to use

1. **Read** each markdown cell, then **run** the code beneath it (`Shift+Enter`).
2. **Change parameters** and re-run — statistics is about *relationships*, not memorized formulas.
3. LaTeX uses \( \) for inline math and \[ \] / $$ $$ for display math (KaTeX-friendly).

Let's begin.


---
## Setup — run this first

NumPy for numerics, SciPy for exact distributions, Matplotlib for plots.


In [ ]:
# If anything is missing, uncomment and run once:
# !pip install numpy scipy matplotlib

import numpy as np
import matplotlib.pyplot as plt
import scipy
from scipy import stats

np.random.seed(42)
rng = np.random.default_rng(42)
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True

print("Ready. NumPy", np.__version__, "| SciPy", scipy.__version__)


## Part 1 — Probability space and the three axioms

A **probability space** is the triple \((\Omega, \mathcal{F}, P)\):

- **Sample space** \(\Omega\): everything that could happen.
- **Sigma-algebra** \(\mathcal{F}\): the collection of measurable *events*.
- **Probability measure** \(P\): assigns each event \(A \in \mathcal{F}\) a number \(P(A) \in [0, 1]\).

\(P\) must satisfy three axioms:

1. **Non-negativity:** \(P(A) \geq 0\) for every event \(A\).
2. **Normalization:** \(P(\Omega) = 1\).
3. **Countable additivity:** for disjoint \(A_1, A_2, \dots\),
   \[
   P\!\left(\bigcup_{i=1}^{\infty} A_i\right) = \sum_{i=1}^{\infty} P(A_i).
   \]

For a finite sample space with equally likely outcomes,
\[
P(A) = \frac{|\{\omega \in A\}|}{|\Omega|}.
\]

Below we build a tiny probability space — rolling a fair six-sided die — and check each axiom numerically.


In [ ]:
# A finite probability space: fair six-sided die.
Omega = np.arange(1, 7)                 # {1,2,3,4,5,6}
P_face = np.full(6, 1/6)                # equally likely outcomes

# Axiom 1: non-negativity
assert np.all(P_face >= 0), "Axiom 1 violated"

# Axiom 2: normalization — total probability mass = 1
total_mass = P_face.sum()
print(f"sum of P over Omega = {total_mass:.6f}")
assert np.isclose(total_mass, 1.0)

# Axiom 3: finite additivity for disjoint singletons A={2}, B={4}
A, B = np.array([2]), np.array([4])
pA = P_face[Omega == 2].sum()
pB = P_face[Omega == 4].sum()
pA_union_B = P_face[np.isin(Omega, np.union1d(A, B))].sum()
print(f"P(A)={pA:.4f}, P(B)={pB:.4f}, P(A u B)={pA_union_B:.4f}")
assert np.isclose(pA_union_B, pA + pB)
print("All three axioms hold for this probability space.")


## Part 2 — Equally likely outcomes, complement, and the addition rule

When outcomes are equally likely, probability is "favorable over total":

\[
P(A) = \frac{|A|}{|\Omega|}.
\]

Two derived rules used everywhere:

- **Complement rule:** \(P(A^c) = 1 - P(A)\).
- **Addition rule (inclusion–exclusion for two events):**
  \[
  P(A \cup B) = P(A) + P(B) - P(A \cap B).
  \]

The subtraction matters: the overlap \(A \cap B\) is counted in *both* \(P(A)\) and \(P(B)\), so we remove one copy.

We verify both rules with a Monte Carlo estimate of the football/basketball example from the PDF (\(P(A)=0.4\), \(P(B)=0.3\), \(P(A \cap B)=0.1\)).


In [ ]:
rng = np.random.default_rng(42)
N = 200_000

# Football / basketball example from the PDF.
pA, pB, pAB = 0.4, 0.3, 0.1

# Analytic results
pA_union_B = pA + pB - pAB
print(f"Analytic: P(A u B) = {pA_union_B:.4f}, P(A^c) = {1 - pA:.4f}")

# Joint simulation: draw (football, basketball) outcomes with the right marginals
# and intersection by using a 2x2 table.
pB_given_A = pAB / pA          # P(B | A)
pB_given_Ac = (pB - pAB) / (1 - pA)   # P(B | A^c)

plays_football = rng.random(N) < pA
plays_basket = np.where(
    plays_football,
    rng.random(N) < pB_given_A,
    rng.random(N) < pB_given_Ac,
)

pA_hat = plays_football.mean()
pB_hat = plays_basket.mean()
pAB_hat = (plays_football & plays_basket).mean()
pA_union_B_hat = (plays_football | plays_basket).mean()
pAc_hat = (~plays_football).mean()

print(f"Sim:     P(A)={pA_hat:.4f}  P(B)={pB_hat:.4f}  "
      f"P(A n B)={pAB_hat:.4f}  P(A u B)={pA_union_B_hat:.4f}  "
      f"P(A^c)={pAc_hat:.4f}")
print(f"Addition rule check: P(A)+P(B)-P(A n B) = "
      f"{pA_hat + pB_hat - pAB_hat:.4f}  vs  P(A u B) = {pA_union_B_hat:.4f}")


## Part 3 — Conditional probability and independence

The **conditional probability** of \(A\) given \(B\) (with \(P(B)>0\)) is

\[
P(A \mid B) = \frac{P(A \cap B)}{P(B)}.
\]

It is the *relative frequency* of \(A\) inside the world where \(B\) already happened.

Two events \(A, B\) are **independent** when

\[
P(A \cap B) = P(A)\,P(B),
\]

equivalently \(P(A \mid B) = P(A)\): knowing \(B\) tells you nothing about \(A\).

We reproduce the PDF's coin + die example by simulation, comparing the conditional
frequency \(P(A \mid B)\) against the formula, and checking the product definition
of independence.


In [ ]:
rng = np.random.default_rng(42)
N = 600_000

# A = coin shows heads, B = die shows 6.
coin = rng.integers(0, 2, size=N)          # 0 tails, 1 heads
die  = rng.integers(1, 7, size=N)           # 1..6

A = coin == 1
B = die == 6

pA = A.mean()
pB = B.mean()
pAB = (A & B).mean()
pA_given_B = (A & B).sum() / B.sum()

print(f"P(A)     = {pA:.4f}  (expected 0.5)")
print(f"P(B)     = {pB:.4f}  (expected 1/6 = {1/6:.4f})")
print(f"P(A n B) = {pAB:.4f}  (expected 1/12 = {1/12:.4f})")
print(f"P(A|B)   = {pA_given_B:.4f}  (formula = P(A n B)/P(B) = {pAB/pB:.4f})")
print(f"Independence: P(A)*P(B) = {pA*pB:.4f}  vs  P(A n B) = {pAB:.4f}")
assert np.isclose(pA_given_B, pA, atol=0.01)
print("=> A and B are independent (conditional equals marginal).")


## Part 4 — Bayes' theorem

**Bayes' theorem** flips the direction of conditioning:

\[
P(A \mid B) = \frac{P(B \mid A)\,P(A)}{P(B)}
   = \frac{P(B \mid A)\,P(A)}{P(B \mid A)P(A) + P(B \mid A^c)P(A^c)}.
\]

The denominator is the **law of total probability**: it averages the likelihood
over the prior. The result is the **posterior** \(P(A \mid B)\) — your updated
belief after seeing evidence \(B\).

The PDF's medical-test example:

- Prevalence \(P(D) = 0.01\).
- Sensitivity \(P(+ \mid D) = 0.99\).
- False-positive rate \(P(+ \mid D^c) = 0.05\).

Gives \(P(D \mid +) \approx 0.1667\). We simulate the population and check it.


In [ ]:
rng = np.random.default_rng(42)
N = 1_000_000

# Prior and test characteristics
pD = 0.01
pPos_given_D = 0.99
pPos_given_Dc = 0.05

# Generate disease status, then test result conditional on disease status.
has_disease = rng.random(N) < pD
test_positive = np.where(
    has_disease,
    rng.random(N) < pPos_given_D,
    rng.random(N) < pPos_given_Dc,
)

# Analytic Bayes
pPos = pPos_given_D * pD + pPos_given_Dc * (1 - pD)
posterior_analytic = (pPos_given_D * pD) / pPos
print(f"Analytic P(+)        = {pPos:.6f}")
print(f"Analytic P(D | +)    = {posterior_analytic:.4f}")

# Empirical posterior: among positive tests, fraction that actually have D
posterior_emp = has_disease[test_positive].mean()
print(f"Empirical P(D | +)   = {posterior_emp:.4f}")

print(f"\nA positive test raises P(D) from {pD:.2f} to ~{posterior_emp:.2%}, "
      "but the disease is still unlikely because it is rare and false positives pile up.")


---

**Your turn:** In the medical-test example above, leave the test characteristics
(sensitivity \(0.99\), false-positive \(0.05\)) fixed but raise the prevalence
\(P(D)\) to \(0.10\), then \(0.50\). How does \(P(D \mid +)\) change? Re-run the
simulation and compare to Bayes' formula. Try to predict the answer *before* you
run it — this is the single most useful habit when working with conditional
probability.


---
# Problem Set

**Try each problem before revealing the solution.**


## Problems

1. A fair die is rolled once. Let \(A =\) "roll is even", \(B =\) "roll is \(> 3\)".

(a) Compute \(P(A)\), \(P(B)\), \(P(A \cap B)\) using the equally-likely-outcomes formula.
(b) Use the addition rule to find \(P(A \cup B)\).
(c) Verify all three numbers with a NumPy simulation of 500,000 rolls.

2. In a company, 40% of employees are in IT and 20% are in IT *and* have a PhD.
Find \(P(	ext{PhD} \mid 	ext{IT})\) by formula, then estimate it by simulating
1,000,000 employees using the joint construction from Part 2 (i.e. draw
`in_IT = rng.random(N) < 0.4`, then `has_PhD = rng.random(N) < (0.2/0.4)` *restricted*
to those in IT). Compare the simulation to the exact value 0.5.

3. Two fair coins are tossed. Let \(A =\) "first coin is heads", \(B =\) "the two
coins match".

(a) List \(\Omega\), enumerate \(A\) and \(B\), and compute \(P(A)\), \(P(B)\), \(P(A \cap B)\).
(b) Are \(A\) and \(B\) independent? Check both the product definition and the
\(P(A \mid B) = P(A)\) characterization.
(c) Confirm with a simulation of 1,000,000 pairs of tosses.

4. A spam filter has: \(P(	ext{spam}) = 0.3\), \(P(	ext{flag} \mid 	ext{spam}) = 0.98\),
\(P(	ext{flag} \mid 	ext{not spam}) = 0.04\).

(a) Use the law of total probability to compute \(P(	ext{flag})\).
(b) Use Bayes' theorem to compute \(P(	ext{spam} \mid 	ext{flag})\).
(c) Simulate 1,000,000 emails and confirm the analytic posterior empirically.

5. For two events \(A, B\) in the same probability space, show numerically that
inclusion–exclusion extends to three events:

\[
P(A \cup B \cup C) = P(A) + P(B) + P(C) - P(A \cap B) - P(A \cap C) - P(B \cap C) + P(A \cap B \cap C).
\]

Pick \(P(A)=0.5\), \(P(B)=0.4\), \(P(C)=0.3\) with pairwise intersections
\(P(A \cap B)=0.2\), \(P(A \cap C)=0.15\), \(P(B \cap C)=0.1\), and
\(P(A \cap B \cap C)=0.05\). Compute the union both ways (formula and direct
intersection probability) and confirm they agree.


<details>
<summary><strong>Reveal solutions</strong></summary>

**1.** \(\Omega = \{1,2,3,4,5,6\}\), \(|A|=|\{2,4,6\}|=3\), \(|B|=|\{4,5,6\}|=3\),
\(A \cap B = \{4,6\}\) so \(|A \cap B|=2\).

(a) \(P(A) = 3/6 = 0.5\), \(P(B) = 3/6 = 0.5\), \(P(A \cap B) = 2/6 \approx 0.3333\).
(b) \(P(A \cup B) = 0.5 + 0.5 - 1/3 = 5/6 \approx 0.8333\).
(c) Sim: `rolls = rng.integers(1,7,N); A = rolls%2==0; B = rolls>3`. You should
see \(P(A \cup B) \approx 0.833\).

```python
rng = np.random.default_rng(0)
N = 500_000
rolls = rng.integers(1, 7, N)
A = (rolls % 2 == 0)
B = (rolls > 3)
print((A | B).mean())   # ~0.833
```

**2.** \(P(\text{PhD} \mid \text{IT}) = 0.2 / 0.4 = 0.5\). Simulation:

```python
rng = np.random.default_rng(0)
N = 1_000_000
in_IT = rng.random(N) < 0.4
# 50% of those in IT have a PhD:
has_PhD = np.zeros(N, dtype=bool)
has_PhD[in_IT] = rng.random(in_IT.sum()) < 0.5
print(has_PhD[in_IT].mean())   # ~0.5
```

**3.** \(\Omega = \{HH, HT, TH, TT\}\), each with probability \(1/4\).
\(A = \{HH, HT\}\), \(B = \{HH, TT\}\), \(A \cap B = \{HH\}\).

(a) \(P(A)=2/4=0.5\), \(P(B)=2/4=0.5\), \(P(A \cap B)=1/4=0.25\).
(b) \(P(A)P(B) = 0.25 = P(A \cap B)\), so independent. Also
\(P(A \mid B) = 0.25/0.5 = 0.5 = P(A)\). Independent both ways.

```python
rng = np.random.default_rng(0)
N = 1_000_000
c1 = rng.integers(0, 2, N)
c2 = rng.integers(0, 2, N)
A = c1 == 1
B = c1 == c2
print(A.mean(), B.mean(), (A & B).mean(), (A & B).mean() / B.mean())
```

**4.** (a) \(P(\text{flag}) = 0.98 \cdot 0.3 + 0.04 \cdot 0.7 = 0.294 + 0.028 = 0.322\).
(b) \(P(\text{spam} \mid \text{flag}) = 0.294 / 0.322 \approx 0.9130\).

```python
rng = np.random.default_rng(0)
N = 1_000_000
spam = rng.random(N) < 0.3
flag = np.where(spam, rng.random(N) < 0.98, rng.random(N) < 0.04)
print(spam[flag].mean())   # ~0.913
```

**5.** Analytic:
\(0.5 + 0.4 + 0.3 - 0.2 - 0.15 - 0.1 + 0.05 = 0.8\).
Build the 7 non-empty regions of the Venn diagram with probabilities
\(p_{ABC} = 0.05\), \(p_{AB}=0.15\), \(p_{AC}=0.10\), \(p_{BC}=0.05\),
\(p_A = 0.5 - 0.15 - 0.10 - 0.05 = 0.20\), \(p_B = 0.4 - 0.15 - 0.05 - 0.05 = 0.15\),
\(p_C = 0.3 - 0.10 - 0.05 - 0.05 = 0.10\). Sum of all seven atoms = 0.8.

```python
pABC = 0.05; pAB = 0.15; pAC = 0.10; pBC = 0.05
pA = 0.5 - pAB - pAC - pABC
pB = 0.4 - pAB - pBC - pABC
pC = 0.3 - pAC - pBC - pABC
union_formula = pA + pB + pC + pAB + pAC + pBC + pABC
union_direct = 0.5 + 0.4 + 0.3 - 0.2 - 0.15 - 0.1 + 0.05
print(union_formula, union_direct)   # 0.8, 0.8
```

</details>
